Phase 4 — Common Table Expressions (CTE)

=====================================================================================

Q049 ⭐⭐ (Easy)
Business Scenario

The HR manager asks:

"I frequently need to reuse the list of active employees. I don't want to repeat the WHERE is_active = 'Y' condition in every query."

Your task is to create a CTE named active_employees and use it to display:

Employee Name,
Department Name,
Salary

In [0]:
%sql
with active_employees as (
    select e.*
        from sql_interview.employees e
            where is_active = 'Y'
)
select ae.emp_name as employee_name, d.dept_name as department_name, ae.salary
    from active_employees ae
        inner join sql_interview.departments d
            on ae.dept_id = d.dept_id

Q050 ⭐⭐⭐ (Multiple CTEs)

Now we'll use multiple CTEs, which is a very common pattern in Data Engineering pipelines.

Business Scenario

The HR team wants a report showing employees who earn more than the average salary of their own department.

Requirements

Create two CTEs:

1. active_employees

    * Active employees only.

2. department_avg_salary

    * Department ID

    * Average salary

Then return:

Employee Name,
Department Name,
Salary,
Department Average Salary

Only include employees whose:
salary > department_average_salary

Rules: 
✅ Use exactly two CTEs
✅ Join the CTEs
❌ No window functions (even though they could solve it)
❌ No subqueries

This introduces the common pattern of building one logical transformation per CTE and then composing them in the final query.

In [0]:
%sql

with active_employees as (
    select e.emp_name, e.dept_id, e.salary
        from sql_interview.employees e
    where e.is_active = 'Y'
),
department_avg_salary as (
    select dept_id, cast(avg(salary) as decimal(10,2)) as department_avg_salary
        from active_employees
            group by dept_id
)
select ae.emp_name as employee_name, d.dept_name as department_name, ae.salary as salary, das.department_avg_salary
    from active_employees ae
        inner join department_avg_salary das
            on ae.dept_id = das.dept_id
                and  ae.salary > das.department_avg_salary
        
        inner join sql_interview.departments d
            on ae.dept_id = d.dept_id


Q051 — CTE + Window Functions ⭐⭐⭐⭐

This question combines everything you've learned so far.

Business Scenario

The HR manager wants to identify the top 2 highest-paid employees in each department, considering only active employees.

Requirements

Create two CTEs:

1. active_employees
2. ranked_employees (using DENSE_RANK())

Return:

Employee Name,
Department Name,
Salary,
Salary Rank

Only return employees with:

* salary_rank <= 2

Rules:
✅ Two CTEs
✅ Use DENSE_RANK()
✅ No subqueries
❌ No QUALIFY clause (Databricks supports it, but many databases do not, and we're practicing portable SQL).

In [0]:
%sql
with active_employees as (
    select emp_id, emp_name, dept_id, salary
        from sql_interview.employees
    where is_active = 'Y'
),
ranked_employees as (
    select ae.emp_id, ae.emp_name, ae.dept_id, ae.salary,
            dense_rank() over(partition by ae.dept_id order by ae.salary desc) as salary_rank
        from active_employees ae
)
select re.emp_name as employee_name, d.dept_name as department_name, re.salary as salary, re.salary_rank 
    from ranked_employees re
        inner join sql_interview.departments d
            on re.dept_id = d.dept_id
    where re.salary_rank <= 2


Q052 ⭐⭐⭐⭐ (Multiple CTEs + Aggregation + Window Function)
Business Scenario

The Sales Director wants to identify the highest-paying department in the company.

However, instead of showing just one department, they want to see the Top 3 departments based on average salary

Requirements:

Write the query using CTEs.

Step 1:

Create a CTE that calculates:

* Department ID
* Department Average Salary

(Consider only active employees.)

Step 2:

Create another CTE that ranks departments by

department_average_salary DESC

using:
* DENSE_RANK()

Alias:
* department_rank

Final Output:

Return:

* Department Name
* Department Average Salary
* Department Rank


Only return:
* department_rank <= 3

Sort by:
* department_rank,
* department_average_salary DESC

Rules:
* ✅ Exactly 2 CTEs
* ✅ Use DENSE_RANK()
* ✅ Join with departments
* ❌ No subqueries
❌ No QUALIFY

In [0]:
%sql

with active_department_avg as (
    select dept_id, avg(salary) as department_avg_salary
        from sql_interview.employees
    where is_active = 'Y'
        and dept_id is not null
    group by dept_id
),
dept_rank_cal as (
    select dept_id, department_avg_salary,
            dense_rank() over(order by department_avg_salary desc) department_rank
        from active_department_avg
)
select d.dept_name as department_name, cast(dr.department_avg_salary as decimal(10,2)) as department_avg_salary, dr.department_rank 
    from dept_rank_cal dr
        inner join sql_interview.departments d
            on dr.dept_id = d.dept_id
    where dr.department_rank <= 3
    order by department_rank asc,
            department_avg_salary desc
